# M4c - baseline on the 7B target

Same battery, bigger victim: **Qwen2.5-7B-Instruct** instead of 3B.

**The prediction we're testing.** On the 3B, most attacks failed for *capability*
reasons - it couldn't do base64/rot13 (gibberish out), and it couldn't hold a persona
frame like AIM, so it just refused. Only the low-effort attacks worked
(prefix_injection 96%, distractors 48%, wikipedia_article 42%).

A 7B should be capable enough to actually *follow* those instructions - so the cipher
and persona attacks should climb. If they do, that's a clean size-vs-jailbreakability
result for the report.

**GPU layout** (needs `GPU T4 x2`):
- target 7B in **fp16**, sharded across both cards (~15 GB; too big for one 16 GB T4)
- judge 9B in 4-bit on cuda:1 (~6 GB)

No quantization on the victim, so there's no 'we attacked a squashed model' caveat.

Expect **~2-4 h** for the full 900 trials. Run the 3-goal sanity first.

## 1 - Setup

In [ ]:
%pip -q install -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
import os, subprocess, sys, pathlib, time, json, glob, shutil
import numpy as np, pandas as pd

_sec = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
except Exception as e:
    print('no Kaggle secrets client:', e)

def _secret(name):
    try:
        return _sec.get_secret(name) if _sec is not None else None
    except Exception:
        return None

_hf = _secret('HF_TOKEN')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    from huggingface_hub import login; login(token=_hf)
    print('HF auth OK')
else:
    print('no HF_TOKEN secret (fine - models are public)')

In [ ]:
# --- get the repo -----------------------------------------------------------
REPO   = "MehemudAzad/LLM-jailbreaking-with-layered-prompt-defense"
BRANCH = "main"
WORK   = pathlib.Path("/kaggle/working")
ROOT   = WORK / "repo"

_gh  = _secret("GH_TOKEN")
_url = f"https://{_gh}@github.com/{REPO}.git" if _gh else f"https://github.com/{REPO}.git"

os.chdir(WORK)
subprocess.run(["rm", "-rf", str(ROOT)], check=False)
_r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, _url, str(ROOT)],
                    cwd=str(WORK), capture_output=True, text=True)
if _r.returncode != 0:
    _err = _r.stderr.replace(_gh, "***") if _gh else _r.stderr
    raise RuntimeError("git clone failed:\n" + _err)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("HEAD", subprocess.check_output(["git","-C",str(ROOT),"rev-parse","--short","HEAD"]).decode().strip())

_missing = [f for f in ("run_eval.py", "report.py", "core/models.py", "datasets/build_harmful.py")
            if not (ROOT / f).exists()]
if _missing:
    raise RuntimeError(f"clone is missing {_missing} -- commit and push them, then re-run this cell")
print("repo files OK")

In [ ]:
r = subprocess.run([sys.executable, "datasets/build_harmful.py"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

In [ ]:
import torch
print('CUDA devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {p.total_memory/1e9:.0f} GB')
assert torch.cuda.device_count() >= 2, 'set the accelerator to GPU T4 x2 -- the 7B needs both'

In [ ]:
from core.config import CONFIG
for role in ('target', 'helper', 'judge'):
    s = CONFIG['models'][role]
    print(f"{role:<8} {s['name']:<26} device={s.get('device'):<8} quant={s.get('quant')}"
          f"  max_memory={s.get('max_memory')}")

## 2 - Load the target and check the split

`device = "auto"` + `max_memory` shards the 7B across both cards while leaving room on
cuda:1 for the judge. This cell confirms it actually landed that way.

In [ ]:
from core.seed import seed_everything
from core.models import load_target

seed_everything()
t0 = time.time()
target = load_target()
print(target.generate('Say hello in five words.', max_new_tokens=20))
print(f'loaded in {time.time()-t0:.0f}s')

for i in range(torch.cuda.device_count()):
    print(f'  cuda:{i}  {torch.cuda.memory_allocated(i)/1e9:.1f} GB used')

model = target._bundle()[0]
places = sorted({str(p.device) for p in model.parameters()})
print('target is on:', places, '<- both means sharding worked')

## 3 - Sanity pass (3 goals)

~54 trials. Loads the judge too, so watch that cuda:1 has room for both.

In [ ]:
from run_eval import main
main(['--attack', 'all', '--defense', 'off', '--limit', '3', '--tag', 'm4c7bsanity'])

In [ ]:
import report
report.print_asr('m4c7bsanity', title='SANITY 7B (3 goals, undefended)')
for i in range(torch.cuda.device_count()):
    print(f'cuda:{i}  {torch.cuda.memory_allocated(i)/1e9:.1f} GB')

In [ ]:
report.samples('m4c7bsanity', exclude='GOOD_BOT', n=8)

### Gate

1. No out-of-memory error, and cuda:1 holds both the judge and the target's slice.
2. Cipher attacks show `judge:decoded` with **readable** plaintext this time - a 7B
   should encode properly, unlike the 3B's word salad.
3. Labels match the replies.

If cuda:1 is near full, lower `max_memory` for GPU 1 in `config.toml` (say `"5GiB"`) so
more of the target sits on cuda:0.

## 4 - Full baseline (50 goals, ~2-4 h)

In [ ]:
main(['--attack', 'all', '--defense', 'off', '--tag', 'm4c7bbaseline'])

In [ ]:
report.print_asr('m4c7bbaseline', title='BASELINE ASR - undefended Qwen2.5-7B, 50 AdvBench goals')

## 5 - 3B vs 7B

Needs the old 3B run attached (the `baseline-asr-llm-jailbreak` dataset). Skipped if absent.

In [ ]:
old = [pathlib.Path(p).parent for p in
       glob.glob('/kaggle/input/**/transcript.jsonl', recursive=True)
       if 'm4baseline' in p]
if old:
    report.print_compare(str(old[0]), 'm4c7bbaseline', label_a='3B', label_b='7B')
else:
    print('3B run not attached -- add the baseline-asr-llm-jailbreak dataset to compare')

## 6 - Save + pin

In [ ]:
from huggingface_hub import HfApi
for role in ('target', 'judge'):
    nm = CONFIG['models'][role]['name']
    try:
        sha = HfApi().model_info(nm, token=os.environ.get('HF_TOKEN')).sha
        print(f'[models.{role}]  # {nm}\n  revision = "{sha}"')
    except Exception as e:
        print(role, 'lookup failed:', e)

In [ ]:
!cd /kaggle/working && zip -qr artifacts_7b.zip artifacts && ls -la artifacts_7b.zip
print()
!ls /kaggle/working/artifacts

## Done

Download `artifacts_7b.zip` into your local `logs/`, and paste the printed `revision`
values into `config.toml` (target is currently `PIN-ME`).

**What the comparison tells you:** if the cipher and persona attacks jumped, the 3B's low
ASR really was a capability ceiling, and model size is a variable worth a section in the
report. If they stayed flat, then Qwen's safety training is doing the work regardless of
size - also a fine result, just a different one.

**Next: M5** - Layer 2 paraphraser, then the defended run against whichever target you keep.